In [ ]:
# =========================================================
# Fine-tuning XLNet (xlnet-base-cased) for English
# AI-Generated Text Detection — DOCUMENT-LEVEL TRAINING
# (see chat, 2026-10-17): mirrors the DeBERTa-v3 document-level fine-tuning
# script exactly, with XLNet (xlnet-base-cased) swapped in as the
# backbone. Each document is tokenized and fed to the model as ONE example
# (its own label), with NO sentence splitting for training or inference —
# there is no sentence-level anywhere in this script; every prediction,
# metric, confusion matrix, threshold search and plot is already at the
# document level by construction.
# (see chat) — uses the SAME normalization pipeline as every other English
# baseline (shared text across all baselines for a fair, identical-input
# comparison), matching the fairness-over-optimality principle established
# for the whole English track. The normalization pipeline re-attaches
# sentence-ending punctuation (. ! ?) directly to the previous word (no
# stray space before it), since earlier steps (citation/URL/email/emoji
# removal, contraction expansion) could leave a dangling space before a
# sentence-ender, e.g. "the cat . " instead of "the cat.".
# Training is kept in full FP32 for the model weights (explicit
# model.float() cast) with CLASS_WEIGHTS' dtype matched to logits in
# WeightedTrainer.compute_loss, and bf16 mixed precision enabled for speed
# on GPU — the same safe, confirmed-working configuration used for every
# other English backbone in this project. If training ever gets stuck at
# random-guessing loss (flat at ln(2)) on this backbone, set bf16=False
# below to fall back to full FP32 training.
# NOTE: XLNet's tokenizer requires the `sentencepiece` package —
# run `!pip install -q sentencepiece` in its own cell first if the
# tokenizer load below fails with a missing-dependency error.
# Binary classification: 0 = Human | 1 = AI-generated
# Split: 80% train / 10% validation / 10% test (stratified by label + domain)
# Evaluation: Accuracy, F1, Precision, Recall, Confusion Matrix, ROC Curve,
#             domain-wise breakdown, and error analysis — ALL document-level.
#
# PUBLICATION-QUALITY EDITION (Q1 journal figures)
#  - Fonts embedded correctly (Type 42/TrueType, not Type 3) for PDF/EPS
#  - Colorblind-safe, consistent palette (Okabe–Ito) across ALL figures
#  - Every figure saved as: 600-dpi PNG (raster, for quick preview / Word)
#                            + vector PDF (for LaTeX / final journal submission)
#  - No in-figure titles (journals put captions in the manuscript text) —
#    controlled by SHOW_TITLES flag below if you still want them while drafting
#  - Standardized figure sizes matching common journal column widths
#  - Larger, consistent tick/label/legend font sizes that stay legible when
#    the figure is shrunk to fit a column
# =========================================================

!pip install -q -U transformers datasets scikit-learn matplotlib seaborn openpyxl

# IMPORTANT (Colab): after this cell installs/upgrades matplotlib, go to
#   Runtime > Restart session
# and THEN run the rest of the notebook from the top (this cell can be
# skipped on the re-run). Importing matplotlib in the same kernel session
# right after upgrading it can leave a stale `font_manager` module cached,
# which later breaks PDF export with:
#   ImportError: cannot import name 'FontPath' from 'matplotlib.font_manager'
# A restart forces the new version to load cleanly and avoids this entirely.

# ──────────────────────────────────────────
# Mount Google Drive first — we read the data and save everything there
# ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.font_manager as fm
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, roc_curve, auc, classification_report,
    matthews_corrcoef, silhouette_score
)
from sklearn.manifold import TSNE
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from datasets import Dataset
import unicodedata

# ──────────────────────────────────────────
# Config
# ──────────────────────────────────────────
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/english_ai_detector_project"
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)

# (see chat, 2026-10-17) — dedicated subfolder for this backbone's
# document-level fine-tuning variant, mirroring the DeBERTa-v3 document-level
# script's own dedicated subfolder convention so tables/plots/model never
# collide with any other backbone's or method's outputs.
METHOD_DIR = f"{DRIVE_PROJECT_DIR}/XLNet_DocumentLevel_FineTune"
os.makedirs(METHOD_DIR, exist_ok=True)

TABLES_DIR = f"{METHOD_DIR}/tables"
PLOTS_DIR  = f"{METHOD_DIR}/plots"
PLOTS_VECTOR_DIR = f"{PLOTS_DIR}/vector"   # PDF/EPS versions for journal submission
os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(PLOTS_VECTOR_DIR, exist_ok=True)

# ── FIXED SPLIT FILES (pre-split, shared across ALL English model scripts
# for a fair comparison — produced by create_fixed_splits_english.py) ──
TRAIN_SPLIT_FILE = "/content/drive/MyDrive/english_ai_detector_project/splits/train_split.csv"
VAL_SPLIT_FILE   = "/content/drive/MyDrive/english_ai_detector_project/splits/val_split.csv"
TEST_SPLIT_FILE  = "/content/drive/MyDrive/english_ai_detector_project/splits/test_split.csv"
MODEL_NAME        = "xlnet-base-cased"   # XLNet
OUTPUT_DIR         = "/content/checkpoints_xlnet_english_doclevel"   # training checkpoints — local (faster)
FINAL_MODEL_DIR    = f"{METHOD_DIR}/model_final_xlnet_doclevel"   # final model — saved to Drive

DOMAIN_COLUMN     = "domain"   # column with values e.g. "news" / "academics" — change if different in your file

# (see chat, 2026-10-17) — MAX_LENGTH is at the whole-document level (not
# individual sentences), so sequences are much longer than the
# sentence-level baselines. 512 is used here for consistency with every other backbone in this project (XLNet itself supports much longer sequences via relative positional encoding, but 512 keeps the comparison fair).
MAX_LENGTH    = 512
# (see chat, 2026-10-17) — BATCH_SIZE kept at 32: with MAX_LENGTH=512 and
# full documents (vs. short sentences in the sentence-level baselines), a
# much larger batch would risk GPU OOM. 32 at seq_len=512 on
# xlnet-base; lower it to 16 if this OOMs on your GPU.
BATCH_SIZE    = 32
EPOCHS        = 5
LEARNING_RATE = 2e-5   # see chat — standard rate, same as every other
                        # English classifier in this project.
RANDOM_SEED   = 42

SHOW_TITLES = False   # True while drafting; set False for the final journal figures
                       # (most Q1 journals want the caption in the manuscript text,
                       #  not baked into the image)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device in use: {device}")
if device == "cpu":
    print("WARNING: No GPU available! Training will be very slow.")
    print("   Go to Runtime > Change runtime type > select GPU (T4)")

# ──────────────────────────────────────────
# PUBLICATION STYLE CONFIG — applies to every figure below
# ──────────────────────────────────────────
# 1) Embed fonts properly instead of Type 3 bitmap fonts (many journals/LaTeX
#    reject Type 3 fonts outright).
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype']  = 42

# 2) Font family: try a journal-standard serif first (Times New Roman / Nimbus
#    Roman, close metric match), fall back gracefully if not installed on the
#    machine so the script never crashes.
_preferred_fonts = ['Times New Roman', 'Nimbus Roman', 'Liberation Serif', 'DejaVu Serif']
_available = {f.name for f in fm.fontManager.ttflist}
_font_family = next((f for f in _preferred_fonts if f in _available), 'DejaVu Serif')

plt.rcParams.update({
    'font.family': _font_family,
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'figure.dpi': 150,          # on-screen preview
    'savefig.dpi': 600,         # exported raster resolution (journals typically want >=300, 600 is safe)
    'savefig.bbox': 'tight',
    'axes.linewidth': 0.9,
    'lines.linewidth': 2.0,
})

sns.set_theme(style="white", context="paper", rc=plt.rcParams)

# 3) Okabe–Ito palette: colorblind-safe, standard in Q1 STEM journals.
#    Used for the domain bar charts (error analysis) that are unrelated to
#    the reference dashboard image.
OKABE_ITO = {
    "blue":    "#0072B2",
    "orange":  "#E69F00",
    "green":   "#009E73",
    "red":     "#D55E00",
    "purple":  "#CC79A7",
    "yellow":  "#F0E442",
    "skyblue": "#56B4E9",
    "black":   "#000000",
}

# Colors sampled directly from the reference dashboard image (confusion
# matrix / tables are intentionally excluded — those keep their own
# untouched styling per request). These drive every Human-vs-AI, train-vs-val
# and metric-line color in the plots below so the look matches the reference.
IMG_COLORS = {
    "human":       "#1E90FF",  # dodger blue  — Human class (hist / scatter / lines)
    "ai":          "#FF6347",  # tomato red   — AI class (hist / scatter / lines)
    "roc_line":    "#2E97F0",  # ROC curve line (sampled directly from the image)
    "val_f1":      "#4CAF50",  # green  — Val F1 line
    "val_acc":     "#FF9800",  # orange — Val Accuracy line
}

PALETTE = {"human": IMG_COLORS["human"], "ai": IMG_COLORS["ai"],
           "news": OKABE_ITO["orange"], "academics": OKABE_ITO["green"]}
DOMAIN_COLOR_CYCLE = [OKABE_ITO["blue"], OKABE_ITO["orange"], OKABE_ITO["green"], OKABE_ITO["purple"]]

# 4) Standard journal figure widths (inches) — Elsevier/Springer/IEEE style guides
FIG_SINGLE_COL = (3.5, 3.0)   # single-column figure
FIG_DOUBLE_COL = (7.2, 3.4)   # double-column / wide figure

_pdf_export_warned = False  # avoid spamming the same warning for every single figure

def save_fig(fig, name, tight=True):
    """Save a figure as both a 600-dpi PNG (preview/Word) and a vector PDF (journal submission).

    The PDF export is wrapped in try/except: in Colab, `!pip install -U matplotlib`
    followed by using matplotlib in the SAME kernel session (without a runtime
    restart) can leave a stale `font_manager` module loaded, which breaks
    `backend_pdf`'s import of `FontPath` with an ImportError. If that happens,
    we still keep the PNG (which does not need that backend) and just skip the
    PDF instead of crashing the whole run.
    """
    global _pdf_export_warned
    if tight:
        fig.tight_layout()
    fig.savefig(f"{PLOTS_DIR}/{name}.png", dpi=600, bbox_inches='tight')
    try:
        fig.savefig(f"{PLOTS_VECTOR_DIR}/{name}.pdf", bbox_inches='tight')
    except ImportError as e:
        if not _pdf_export_warned:
            print(f"WARNING: vector PDF export disabled ({e}).")
            print("   This is almost always fixed by: Runtime > Restart session,")
            print("   then re-running the notebook (matplotlib was upgraded mid-session).")
            print("   PNG figures are still being saved normally.")
            _pdf_export_warned = True

def style_ax(ax, ylabel=None, xlabel=None, title=None):
    """Clean, minimal, Q1-journal-style axis: no top/right spine, subtle gridlines."""
    if title and SHOW_TITLES:
        ax.set_title(title, fontsize=14, fontweight='bold', pad=10)
    if ylabel:
        ax.set_ylabel(ylabel)
    if xlabel:
        ax.set_xlabel(xlabel)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#555555')
    ax.spines['bottom'].set_color('#555555')
    ax.grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.35, zorder=0)
    ax.set_axisbelow(True)

# Backwards-compatible alias used lower in the script for bar charts
def style_bar_ax(ax, title, ylabel, xlabel=None):
    style_ax(ax, ylabel=ylabel, xlabel=xlabel, title=title)

# ──────────────────────────────────────────
# 1) Load the 3 FIXED split files (no re-splitting here — the split was
#    already done once, on raw un-preprocessed text, and saved to disk so
#    every model script uses the exact same Train/Val/Test rows)
# ──────────────────────────────────────────
required_cols = ['text', 'label', DOMAIN_COLUMN]

def _load_split(path, name):
    d = pd.read_excel(path) if path.endswith(('.xlsx', '.xls')) else pd.read_csv(path)
    missing = [c for c in required_cols if c not in d.columns]
    if missing:
        raise ValueError(f"Missing required column(s) in {name} split: {missing}")
    d = d[required_cols].dropna().reset_index(drop=True)
    d['label'] = d['label'].astype(int)
    d[DOMAIN_COLUMN] = d[DOMAIN_COLUMN].astype(str).str.strip().str.lower()
    return d

train_df = _load_split(TRAIN_SPLIT_FILE, "train")
val_df   = _load_split(VAL_SPLIT_FILE,   "validation")
test_df  = _load_split(TEST_SPLIT_FILE,  "test")

print(f"\nLoaded fixed splits — Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")
for name, d in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"  {name} label distribution:\n{d['label'].value_counts()}")
    print(f"  {name} domain distribution:\n{d[DOMAIN_COLUMN].value_counts()}")

# ──────────────────────────────────────────
# 2) Preprocess text with the MANUAL normalization pipeline (see chat —
#    agreed English pipeline): remove invisible chars -> remove
#    citations/URLs/emails/emoji -> NFKC -> expand contractions ->
#    remove non-sentence-final punctuation -> re-attach sentence-ending
#    punctuation to the previous word -> collapse whitespace.
#    (applied AFTER loading the fixed splits, separately to each split —
#    preprocessing never influences which rows land in which split anymore)
# ──────────────────────────────────────────
# (see chat) — ENGLISH normalization pipeline, agreed on:
#   1) remove invisible Unicode control characters
#   2) remove citation patterns [1], (Smith, 2020)
#   3) remove URLs and email addresses
#   4) remove emoji / pictographic symbols
#   5) Unicode NFKC normalization (standardizes smart quotes, long dashes, etc.)
#   6) expand common English contractions (don't -> do not, won't -> will not)
#   7) remove non-sentence-final punctuation, collapse whitespace
#      (sentence-ending . ! ? are KEPT)
#   8) (see chat, 2026-10-16, NEW) — re-attach sentence-ending punctuation
#      (. ! ?) directly to the word before it: earlier steps can leave a
#      stray space right before a sentence-ender (e.g. citation/URL/emoji
#      removal replacing a match with a literal space that lands right
#      before a "."), which would otherwise read as "word ." instead of
#      the correct "word.". This runs right before the final whitespace
#      collapse so it catches anything the earlier steps introduced.
_INVISIBLE_CHARS_RE = re.compile('[​‌‍‎‏‪‫‬‭‮﻿]')
_CITATION_RE = re.compile(
    r'\[\d+(?:\s*,\s*\d+)*\]'
    r'|\([^()]{0,60}\d{4}[^()]{0,25}\)'
)
_URL_RE   = re.compile(r'(?:https?://|www\.)\S+')
_EMAIL_RE = re.compile(r'\b[\w.+-]+@[\w-]+\.[\w.-]+\b')
_EMOJI_RE = re.compile(
    "["
    "\U0001F300-\U0001FAFF"
    "\U00002600-\U000027BF"
    "\U0001F1E6-\U0001F1FF"
    "]+", flags=re.UNICODE
)
PUNCTUATION = r";:\"'`“”‘’«»\-–—/\\|@#\$%\^&\*_=~<>\(\)\[\]\{\}"
_NUMERIC_COMMA_SAFE_RE = re.compile(r'(?<!\d),|,(?!\d)')
# (see chat, 2026-10-16, NEW) — collapses any run of whitespace that sits
# directly before a sentence-ending mark into nothing, so "word ." / "word  !"
# / "word\t?" all become "word." / "word!" / "word?".
_SPACE_BEFORE_SENT_END_RE = re.compile(r'\s+([.!?])')

_CONTRACTIONS = {
    "won't": "will not", "can't": "cannot", "n't": " not",
    "'re": " are", "'s": " is", "'d": " would", "'ll": " will",
    "'ve": " have", "'m": " am",
}
_CONTRACTIONS_RE = re.compile("|".join(re.escape(k) for k in _CONTRACTIONS.keys()))

def normalize_arabic_text(text):
    if not isinstance(text, str):
        return text
    text = _INVISIBLE_CHARS_RE.sub('', text)
    text = _CITATION_RE.sub(' ', text)
    text = _URL_RE.sub(' ', text)
    text = _EMAIL_RE.sub(' ', text)
    text = _EMOJI_RE.sub(' ', text)
    text = unicodedata.normalize('NFKC', text)
    text = _CONTRACTIONS_RE.sub(lambda m: _CONTRACTIONS[m.group(0)], text)
    text = _NUMERIC_COMMA_SAFE_RE.sub(" ", text)
    text = re.sub(rf"[{PUNCTUATION}]", " ", text)
    text = _SPACE_BEFORE_SENT_END_RE.sub(r'\1', text)   # NEW — glue ".", "!", "?" to the previous word
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("\nPreprocessing text with the manual normalization pipeline...")
for d in (train_df, val_df, test_df):
    d['text_processed'] = d['text'].astype(str).apply(normalize_arabic_text)

print(f"\nTrain: {len(train_df)}  |  Validation: {len(val_df)}  |  Test: {len(test_df)}")

split_summary = pd.concat([
    train_df.assign(split="train"),
    val_df.assign(split="validation"),
    test_df.assign(split="test"),
])
split_table = (
    split_summary.groupby(['split', DOMAIN_COLUMN, 'label']).size()
    .unstack(fill_value=0).reset_index()
)
print(f"\nSplit x Domain x Label breakdown:\n{split_table}")

# ──────────────────────────────────────────
# 4) Convert to HF Dataset and tokenize — DOCUMENT level
# (see chat, 2026-10-16) — training/validation/test datasets are now built
# directly from the document-level DataFrames (train_df/val_df/test_df).
# There is no sentence expansion anywhere in this script: every row here
# IS one document, with its own label, tokenized as a single (longer)
# sequence up to MAX_LENGTH.
# ──────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(
        batch['text_processed'],
        truncation=True,
        max_length=MAX_LENGTH,
        # NOTE: no padding here — padding='max_length' at this BATCH_SIZE would
        # pad every example to MAX_LENGTH tokens regardless of actual length,
        # wasting memory. Dynamic per-batch padding is handled instead
        # by DataCollatorWithPadding, passed to the Trainer below.
    )

train_dataset = Dataset.from_pandas(train_df[['text_processed', 'label']])
val_dataset   = Dataset.from_pandas(val_df[['text_processed', 'label']])
test_dataset  = Dataset.from_pandas(test_df[['text_processed', 'label']])

train_dataset = train_dataset.map(tokenize_fn, batched=True)
val_dataset   = val_dataset.map(tokenize_fn, batched=True)
test_dataset  = test_dataset.map(tokenize_fn, batched=True)

train_dataset = train_dataset.rename_column("label", "labels")
val_dataset   = val_dataset.rename_column("label", "labels")
test_dataset  = test_dataset.rename_column("label", "labels")

cols = ['input_ids', 'attention_mask', 'labels']
train_dataset.set_format(type='torch', columns=cols)
val_dataset.set_format(type='torch', columns=cols)
test_dataset.set_format(type='torch', columns=cols)

# ──────────────────────────────────────────
# 5) Load model
# ──────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
# (see chat) — explicit FP32 cast: ensures the model is in full precision
# regardless of how the checkpoint was saved, before the mixed-precision
# flags in TrainingArguments take over for training itself.
model = model.float().to(device)
print(f"  Model parameter dtype: {next(model.parameters()).dtype}")

# ──────────────────────────────────────────
# 5b) CLASS WEIGHTS (see chat, 2026-10-16, UPDATED) — now computed on
# DOCUMENT-level labels (train_df), not sentence-level. The fixed splits
# are stratified to be ~50/50 by construction at the document level, so
# these weights will normally land very close to 1.0 each — kept here
# mainly as a safety net in case the document-level split ever drifts
# from exact balance, and to keep the loss function identical in style to
# the rest of this project's scripts.
# ──────────────────────────────────────────
_train_labels_arr = train_df['label'].to_numpy()
_n_human = int((_train_labels_arr == 0).sum())
_n_ai    = int((_train_labels_arr == 1).sum())
CLASS_WEIGHTS = torch.tensor(
    [len(_train_labels_arr) / (2 * _n_human), len(_train_labels_arr) / (2 * _n_ai)],
    dtype=torch.float,
).to(device)
print(f"\nClass weights (document-level): Human={CLASS_WEIGHTS[0]:.3f}  AI={CLASS_WEIGHTS[1]:.3f}")

class WeightedTrainer(Trainer):
    """Overrides the loss to use class-weighted CrossEntropyLoss (see chat)
    instead of the Trainer default's unweighted loss. Class weights are
    matched to logits' dtype/device explicitly, to avoid any dtype mismatch
    between CLASS_WEIGHTS and the logits (e.g. a stray Half vs Float
    conflict) under mixed-precision training."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").long()
        outputs = model(**inputs)
        logits = outputs.logits
        weights = CLASS_WEIGHTS.to(device=logits.device, dtype=logits.dtype)
        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# ──────────────────────────────────────────
# 6) Metrics function used during training (evaluated on validation set)
# (see chat, 2026-10-16, SIMPLIFIED) — eval_pred now gives per-DOCUMENT
# logits/labels directly (val_dataset is already document-level), so no
# sentence->document aggregation is needed anymore; this is a plain,
# direct document-level accuracy/F1/precision/recall.
# ──────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy' : accuracy_score(labels, preds),
        'f1'       : f1_score(labels, preds),
        'precision': precision_score(labels, preds),
        'recall'   : recall_score(labels, preds),
    }

# ──────────────────────────────────────────
# 7) Training arguments
# ──────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_steps=500,   # FIX: was warmup_steps=0.1 — that passed a fraction
                        # (0.1) where an INTEGER step count was expected, which
                        # silently produced ~0 warmup steps. Fixed to a concrete
                        # 500-step warmup instead.
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,
    load_best_model_at_end=False,   # always finish all EPOCHS and keep the
                                     # final (5th) epoch's model; no best-epoch
                                     # selection
    save_total_limit=1,
    report_to="none",
    fp16=False,   # see chat — kept disabled for consistency with the rest of the project
    bf16=torch.cuda.is_available(),   # see chat — enabled for speed on GPU; the
                  # explicit FP32 model load + matched-dtype class weights in
                  # WeightedTrainer (see compute_loss above) keep this safe.
                  # If training ever gets stuck (loss flat at ln(2)), set this
                  # back to False — full FP32 is the confirmed-working fallback.
)

data_collator = DataCollatorWithPadding(tokenizer)   # dynamic per-batch padding
                                                      # (see tokenize_fn note above)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,   # for per-epoch metric logging only (NOT test);
                                 # no checkpoint selection against it
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

# ──────────────────────────────────────────
# 8) Train
# ──────────────────────────────────────────
print("\nStarting training...")
train_result = trainer.train()

# ──────────────────────────────────────────
# 8b) THRESHOLD SEARCH ON VALIDATION (see chat) — try a grid of thresholds
# on the VALIDATION set's DOCUMENT-level P(AI) (now a DIRECT model output —
# no sentence averaging needed) and keep whichever maximizes F1. Applied to
# TEST below — never tuned on test itself. This runs ON TOP OF the class
# weights above (which correct any residual imbalance at its source, during
# training itself) — the two are complementary, not alternatives.
# ──────────────────────────────────────────
print("\nSearching for the best decision threshold on the validation set...")
trainer.compute_metrics = None
val_predictions = trainer.predict(val_dataset)
val_logits = val_predictions.predictions
val_probs_ai = torch.softmax(torch.tensor(val_logits), dim=-1)[:, 1].numpy()
val_doc_labels = val_df['label'].to_numpy()

THRESHOLD_GRID = np.arange(0.40, 0.61, 0.01)   # (see chat, 2026-10-17) narrowed to 0.40-0.60 per user request
threshold_search_rows = []
best_threshold, best_threshold_f1 = 0.5, -1.0
for th in THRESHOLD_GRID:
    preds_th = (val_probs_ai >= th).astype(int)
    f1_th = f1_score(val_doc_labels, preds_th, zero_division=0)
    threshold_search_rows.append({"threshold": round(float(th), 2), "val_f1": round(f1_th, 4)})
    if f1_th > best_threshold_f1:
        best_threshold_f1, best_threshold = f1_th, float(th)

threshold_search_df = pd.DataFrame(threshold_search_rows)
threshold_search_df.to_excel(f"{TABLES_DIR}/threshold_search.xlsx", index=False)
print(f"  Best threshold on VALIDATION (document-level): {best_threshold:.2f}  (val F1={best_threshold_f1:.4f})")
print(f"  (for reference, F1 at fixed 0.5: "
      f"{f1_score(val_doc_labels, (val_probs_ai >= 0.5).astype(int)):.4f})")

fig_th, ax_th = plt.subplots(figsize=FIG_SINGLE_COL)
ax_th.plot(threshold_search_df["threshold"], threshold_search_df["val_f1"],
          color=IMG_COLORS["val_f1"], lw=2)
ax_th.axvline(best_threshold, color="black", ls="--", lw=1, label=f"best={best_threshold:.2f}")
style_ax(ax_th, xlabel="Decision threshold", ylabel="Validation F1 (document-level)", title="Threshold Search (Validation)")
ax_th.legend(fontsize=9)
save_fig(fig_th, "threshold_search_validation")
plt.show()

# ──────────────────────────────────────────
# 9) Final evaluation on the held-out TEST set (never seen during training/tuning)
# (see chat, 2026-10-16, SIMPLIFIED) — prediction is now DIRECTLY at
# document level; there is no sentence-level inference or probability
# averaging step anymore.
# ──────────────────────────────────────────
print("\nRunning final evaluation on the test set (document-level inference)...")
trainer.compute_metrics = None
predictions = trainer.predict(test_dataset)
test_logits = predictions.predictions
test_probs_ai = torch.softmax(torch.tensor(test_logits), dim=-1)[:, 1].numpy()  # P(AI) per document

test_df['prob_ai']    = test_probs_ai
test_df['pred_label'] = (test_df['prob_ai'] >= best_threshold).astype(int)
test_df['true_label'] = test_df['label']
test_df['correct']    = test_df['true_label'] == test_df['pred_label']
test_df['confidence'] = np.where(test_df['pred_label'] == 1, test_df['prob_ai'], 1 - test_df['prob_ai'])

true_labels = test_df['true_label'].values
pred_labels = test_df['pred_label'].values
probs       = test_df['prob_ai'].values

# ──────────────────────────────────────────
# 10) Overall metrics
# ──────────────────────────────────────────
accuracy  = accuracy_score(true_labels, pred_labels)
f1        = f1_score(true_labels, pred_labels)
precision = precision_score(true_labels, pred_labels)
recall    = recall_score(true_labels, pred_labels)
mcc       = matthews_corrcoef(true_labels, pred_labels)

fpr, tpr, _ = roc_curve(true_labels, probs)
roc_auc = auc(fpr, tpr)

print("\n" + "="*55)
print("FINAL RESULTS ON TEST SET (Document-level)")
print("="*55)
print(f"  Threshold : {best_threshold:.2f}  (found via grid search on validation, val F1={best_threshold_f1:.4f})")
print(f"  Accuracy  : {accuracy:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  MCC       : {mcc:.4f}")
print(f"  AUC       : {roc_auc:.4f}")
print("="*55)

overall_metrics_df = pd.DataFrame([{
    'Accuracy': round(accuracy, 4),
    'F1-Score': round(f1, 4),
    'Precision': round(precision, 4),
    'Recall': round(recall, 4),
    'MCC': round(mcc, 4),
    'AUC': round(roc_auc, 4),
}])

report_dict = classification_report(
    true_labels, pred_labels, target_names=['Human (0)', 'AI-generated (1)'], output_dict=True
)
report_df = pd.DataFrame(report_dict).transpose().round(4)
print(f"\nFull classification report:\n{report_df}")

# ──────────────────────────────────────────
# Helper: render a publication-quality table as PNG + PDF
# ──────────────────────────────────────────
def render_table_png(df_table, title, save_name, col_widths=None, figsize=None):
    n_rows, n_cols = df_table.shape
    figsize = figsize or (max(6, n_cols * 1.6), 0.5 * n_rows + 1.0)
    fig, ax = plt.subplots(figsize=figsize)
    ax.axis('off')
    tbl = ax.table(
        cellText=df_table.values,
        colLabels=df_table.columns,
        rowLabels=df_table.index if df_table.index.name or not isinstance(df_table.index, pd.RangeIndex) else None,
        cellLoc='center',
        loc='center',
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(11)
    tbl.scale(1, 1.7)
    for (row, col), cell in tbl.get_celld().items():
        cell.set_edgecolor('#BBBBBB')
        cell.set_linewidth(0.6)
        if row == 0:
            cell.set_facecolor('#1D3557')
            cell.set_text_props(color='white', weight='bold')
        else:
            cell.set_facecolor('#F5F7FA' if row % 2 == 0 else '#FFFFFF')
    if SHOW_TITLES:
        plt.title(title, fontsize=13, fontweight='bold', pad=12)
    save_fig(fig, save_name, tight=True)
    plt.show()

# ──────────────────────────────────────────
# 11) UNIFIED PERFORMANCE TABLE — Overall (document) | per-Domain (document)
# (see chat, 2026-10-16) — the sentence-level rows that existed in the
# sentence-training version are GONE: there is no sentence level anywhere
# in this script anymore.
# ──────────────────────────────────────────
domain_rows = []
for dom in sorted(test_df[DOMAIN_COLUMN].unique()):
    sub = test_df[test_df[DOMAIN_COLUMN] == dom]
    dom_fpr, dom_tpr, _ = roc_curve(sub['true_label'], sub['prob_ai'])
    domain_rows.append({
        'Level':     f'Document — {dom.capitalize()}',
        'N':         len(sub),
        'Accuracy':  round(accuracy_score(sub['true_label'], sub['pred_label']), 4),
        'F1':        round(f1_score(sub['true_label'], sub['pred_label'], zero_division=0), 4),
        'Precision': round(precision_score(sub['true_label'], sub['pred_label'], zero_division=0), 4),
        'Recall':    round(recall_score(sub['true_label'], sub['pred_label'], zero_division=0), 4),
        'MCC':       round(matthews_corrcoef(sub['true_label'], sub['pred_label']), 4),
        'AUC':       round(auc(dom_fpr, dom_tpr), 4),
    })
domain_perf_df = pd.DataFrame(domain_rows)

overall_doc_row = pd.DataFrame([{
    'Level':     'Document — Overall',
    'N':         len(test_df),
    'Accuracy':  round(accuracy, 4),
    'F1':        round(f1, 4),
    'Precision': round(precision, 4),
    'Recall':    round(recall, 4),
    'MCC':       round(mcc, 4),
    'AUC':       round(roc_auc, 4),
}])

unified_df = pd.concat([overall_doc_row, domain_perf_df], ignore_index=True)
unified_df.to_excel(f"{TABLES_DIR}/unified_performance_table.xlsx", index=False)
render_table_png(
    unified_df,
    f"Unified Performance Summary — {MODEL_NAME} (Document-level, Test Set)",
    "table_unified_performance",
    figsize=(13, 0.55 * len(unified_df) + 1.2),
)
print(f"\n  Unified table ({len(unified_df)} rows):")
print(unified_df.to_string(index=False))

# ──────────────────────────────────────────
# 12) CONFUSION MATRICES — Document-level Overall + per-domain
# (see chat, 2026-10-16) — sentence-level panels removed (no sentence level).
# ──────────────────────────────────────────
domains = sorted(test_df[DOMAIN_COLUMN].unique())
n_domain_plots = len(domains)
n_cols = 1 + n_domain_plots
fig_cm, axes_cm = plt.subplots(1, n_cols,
    figsize=(FIG_SINGLE_COL[0] * n_cols, FIG_SINGLE_COL[1] * 1.25))
if n_cols == 1: axes_cm = [axes_cm]

def _plot_cm(ax, cm, title, tn, fp, fn, tp):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                xticklabels=['Human', 'AI'], yticklabels=['Human', 'AI'],
                linewidths=0.5, linecolor='white',
                annot_kws={'fontsize': 13, 'fontweight': 'bold'})
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(title, fontsize=10, fontweight='bold', pad=6)
    ax.text(0.5, -0.18, f"TN={tn}  FP={fp}  FN={fn}  TP={tp}",
            transform=ax.transAxes, ha='center', fontsize=8, color='#444444')

# A) Document-level Overall
cm_doc = confusion_matrix(true_labels, pred_labels)
tn_d, fp_d, fn_d, tp_d = cm_doc[0,0], cm_doc[0,1], cm_doc[1,0], cm_doc[1,1]
_plot_cm(axes_cm[0], cm_doc, 'Document — Overall', tn_d, fp_d, fn_d, tp_d)

# B) Per-domain Document-level
for ax, dom in zip(axes_cm[1:1+n_domain_plots], domains):
    sub = test_df[test_df[DOMAIN_COLUMN] == dom]
    cm_dom = confusion_matrix(sub['true_label'], sub['pred_label'], labels=[0, 1])
    tn_dm, fp_dm, fn_dm, tp_dm = cm_dom[0,0], cm_dom[0,1], cm_dom[1,0], cm_dom[1,1]
    _plot_cm(ax, cm_dom, f'Document — {dom.capitalize()}', tn_dm, fp_dm, fn_dm, tp_dm)

plt.tight_layout()
save_fig(fig_cm, "confusion_matrices_all")
plt.show()

# ──────────────────────────────────────────
# 13) THRESHOLD SEARCH (validation)
# ──────────────────────────────────────────
threshold_search_df.to_excel(f"{TABLES_DIR}/threshold_search.xlsx", index=False)
save_fig(fig_th, "threshold_search_validation")
plt.show()

# ──────────────────────────────────────────
# 14) ROC CURVE
# ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=FIG_SINGLE_COL)
ax.plot(fpr, tpr, color=IMG_COLORS["roc_line"], lw=2.2,
        label=f'{MODEL_NAME} (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], linestyle='--', color='#AAAAAA', lw=1)
ax.set_xlim([-0.02, 1.02]); ax.set_ylim([-0.02, 1.02])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right', fontsize=9)
if SHOW_TITLES:
    ax.set_title('ROC Curve (Test Set)')
save_fig(fig, "roc_curve")
plt.show()

# ──────────────────────────────────────────
# 15) Confidence: correct vs incorrect
# ──────────────────────────────────────────
test_df['correct'] = test_df['pred_label'] == test_df['true_label']
test_df['confidence'] = test_df['prob_ai'].apply(lambda p: p if p >= best_threshold else 1 - p)
fig, ax = plt.subplots(figsize=FIG_SINGLE_COL)
for correct, color, label in [(True, IMG_COLORS["human"], 'Correct'), (False, IMG_COLORS["ai"], 'Incorrect')]:
    subset = test_df[test_df['correct'] == correct]['confidence']
    if len(subset) > 0:
        ax.hist(subset, bins=25, alpha=0.6, color=color, label=label,
                density=True, edgecolor='white', linewidth=0.4)
ax.set_xlabel('Model confidence in predicted class')
ax.set_ylabel('Density')
ax.legend(fontsize=9)
if SHOW_TITLES:
    ax.set_title('Prediction Confidence: Correct vs Incorrect (Document-level)')
save_fig(fig, "confidence_correct_vs_incorrect")
plt.show()

# ──────────────────────────────────────────
# 16) Training loss and validation F1 curves
# ──────────────────────────────────────────
history = trainer.state.log_history
train_loss = [(x['step'], x['loss']) for x in history if 'loss' in x]
eval_f1    = [(x['epoch'], x['eval_f1']) for x in history if 'eval_f1' in x]

if train_loss:
    steps, losses = zip(*train_loss)
    fig, ax = plt.subplots(figsize=FIG_SINGLE_COL)
    ax.plot(steps, losses, color=IMG_COLORS["human"], lw=1.6)
    style_ax(ax, xlabel='Training step', ylabel='Loss', title='Training Loss Curve')
    save_fig(fig, "training_loss")
    plt.show()

if eval_f1:
    epochs_list, f1_list = zip(*eval_f1)
    fig, ax = plt.subplots(figsize=FIG_SINGLE_COL)
    ax.plot(epochs_list, f1_list, marker='o', color=IMG_COLORS["val_f1"], lw=2)
    style_ax(ax, xlabel='Epoch', ylabel='Validation F1-Score', title='Validation F1-Score Across Epochs')
    ax.set_xticks(list(epochs_list))
    save_fig(fig, "f1_progress")
    plt.show()

# ──────────────────────────────────────────
# 17) Combined summary dashboard (single-figure overview, like a model report card)
#     Panels: Confusion Matrix | ROC Curve | Probability Distribution
#             Training Loss    | Validation Metrics | t-SNE Embeddings
# ──────────────────────────────────────────

# 17a) Document embeddings for the t-SNE panel + silhouette score.
#      We mean-pool the last hidden layer (masking out padding tokens) on a
#      random subsample of the test set — 2000 samples keeps t-SNE fast while
#      still giving a representative picture of the embedding space.
# (see chat, 2026-10-16) — TSNE_MAX_LENGTH now matches the training
# MAX_LENGTH (512), since training itself already operates on full
# documents at that length; no reason for the embedding view to truncate
# more aggressively than training did.
N_EMBED_SAMPLES = min(2000, len(test_df))
embed_sample_df = test_df.sample(n=N_EMBED_SAMPLES, random_state=RANDOM_SEED).reset_index(drop=True)

@torch.no_grad()
def extract_embeddings(texts, batch_size=16):
    TSNE_MAX_LENGTH = MAX_LENGTH
    model.eval()
    all_embeds = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        enc = tokenizer(
            batch_texts, padding=True, truncation=True,
            max_length=TSNE_MAX_LENGTH, return_tensors='pt'
        ).to(device)
        out = model(**enc, output_hidden_states=True)
        last_hidden = out.hidden_states[-1]                       # (B, T, H)
        mask = enc['attention_mask'].unsqueeze(-1).float()        # (B, T, 1)
        pooled = (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        all_embeds.append(pooled.cpu().numpy())
    return np.vstack(all_embeds)

print("\nExtracting embeddings for the t-SNE panel...")
embeddings = extract_embeddings(embed_sample_df['text_processed'].tolist())

# Silhouette score computed on the *original* embedding space (not the 2-D
# t-SNE projection) — that's the standard, more meaningful way to measure how
# well-separated the Human/AI clusters are in the model's representation.
sil_score = silhouette_score(embeddings, embed_sample_df['true_label'])

tsne = TSNE(n_components=2, random_state=RANDOM_SEED, perplexity=30, init='pca', learning_rate='auto')
embeddings_2d = tsne.fit_transform(embeddings)

# 17b) Epoch-aggregated train/eval loss and validation metrics from the Trainer log.
train_loss_df = pd.DataFrame(
    [(x['epoch'], x['loss']) for x in history if 'loss' in x], columns=['epoch', 'loss']
)
train_loss_df['epoch_int'] = np.ceil(train_loss_df['epoch']).astype(int)
train_loss_per_epoch = train_loss_df.groupby('epoch_int')['loss'].mean().reset_index()

eval_loss_df = pd.DataFrame(
    [(x['epoch'], x['eval_loss']) for x in history if 'eval_loss' in x], columns=['epoch', 'eval_loss']
)
eval_acc_df = pd.DataFrame(
    [(x['epoch'], x['eval_accuracy']) for x in history if 'eval_accuracy' in x], columns=['epoch', 'eval_accuracy']
)
eval_f1_df = pd.DataFrame(eval_f1, columns=['epoch', 'eval_f1'])

# 17c) Assemble the 2x3 dashboard.
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# -- Panel 1: Confusion Matrix (raw counts only — no percentages) --
ax = axes[0, 0]
sns.heatmap(cm_doc, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=True,
            xticklabels=['Human', 'AI'], yticklabels=['Human', 'AI'],
            linewidths=0.5, linecolor='white', annot_kws={'fontsize': 14, 'fontweight': 'bold'})
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix', fontweight='bold')
ax.text(0.5, -0.16, f"TN={tn_d}   FP={fp_d}   FN={fn_d}   TP={tp_d}",
        transform=ax.transAxes, ha='center', fontsize=9, color='#444444')

# -- Panel 2: ROC Curve --
ax = axes[0, 1]
ax.plot(fpr, tpr, color=IMG_COLORS["roc_line"], lw=2.2, label=f'XLNet (AUC={roc_auc:.4f})')
ax.plot([0, 1], [0, 1], color='#999999', lw=1, linestyle='--')
ax.set_xlabel('FPR')
ax.set_ylabel('TPR')
ax.set_title('ROC Curve', fontweight='bold')
ax.legend(loc='lower right', fontsize=9, frameon=True)

# -- Panel 3: Probability Distribution --
ax = axes[0, 2]
human_probs = test_df.loc[test_df['true_label'] == 0, 'prob_ai']
ai_probs    = test_df.loc[test_df['true_label'] == 1, 'prob_ai']
ax.hist(human_probs, bins=40, alpha=0.6, color=IMG_COLORS["human"], label='Human', density=True)
ax.hist(ai_probs, bins=40, alpha=0.6, color=IMG_COLORS["ai"], label='AI', density=True)
ax.axvline(best_threshold, color='black', linestyle='--', lw=1, label=f'threshold={best_threshold:.2f}')
ax.set_xlabel('P(AI)')
ax.set_title('Probability Distribution', fontweight='bold')
ax.legend(fontsize=8)

# -- Panel 4: Training Loss --
ax = axes[1, 0]
ax.plot(train_loss_per_epoch['epoch_int'], train_loss_per_epoch['loss'],
        marker='o', color=IMG_COLORS["human"], label='Train Loss')
ax.plot(eval_loss_df['epoch'], eval_loss_df['eval_loss'],
        marker='o', color=IMG_COLORS["ai"], label='Val Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss', fontweight='bold')
ax.legend(fontsize=9)

# -- Panel 5: Validation Metrics --
ax = axes[1, 1]
ax.plot(eval_f1_df['epoch'], eval_f1_df['eval_f1'],
        marker='o', color=IMG_COLORS["val_f1"], label='Val F1')
ax.plot(eval_acc_df['epoch'], eval_acc_df['eval_accuracy'],
        marker='s', color=IMG_COLORS["val_acc"], label='Val Accuracy')
ax.set_ylim(0, 1.05)
ax.set_xlabel('Epoch')
ax.set_title('Validation Metrics', fontweight='bold')
ax.legend(fontsize=9)

# -- Panel 6: t-SNE Embeddings --
ax = axes[1, 2]
mask_human = embed_sample_df['true_label'] == 0
mask_ai    = embed_sample_df['true_label'] == 1
ax.scatter(embeddings_2d[mask_human, 0], embeddings_2d[mask_human, 1],
           s=6, color=IMG_COLORS["human"], alpha=0.6, label='Human')
ax.scatter(embeddings_2d[mask_ai, 0], embeddings_2d[mask_ai, 1],
           s=6, color=IMG_COLORS["ai"], alpha=0.6, label='AI')
ax.set_title(f't-SNE Embeddings (n={N_EMBED_SAMPLES})', fontweight='bold')
ax.legend(fontsize=8, markerscale=2)
ax.set_xticks([])
ax.set_yticks([])

# Shared clean styling for the line/scatter/hist panels (skip the two heatmaps)
for ax in [axes[0, 1], axes[0, 2], axes[1, 0], axes[1, 1], axes[1, 2]]:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.3, zorder=0)
    ax.set_axisbelow(True)

fig.suptitle(f"XLNet Document-level Fine-tuning — F1={f1:.4f}   AUC={roc_auc:.4f}   Silhouette={sil_score:.4f}",
             fontsize=15, fontweight='bold', y=1.02)
save_fig(fig, "summary_dashboard")
plt.show()

# ──────────────────────────────────────────
# 18) Save final model
# ──────────────────────────────────────────
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print(f"\nFinal model saved to: {FINAL_MODEL_DIR}")
print(f"Tables saved to: {TABLES_DIR}")
print(f"Plots saved to: {PLOTS_DIR}  (PNG, 600 dpi)")
print(f"Vector plots (PDF, for journal submission) saved to: {PLOTS_VECTOR_DIR}")